In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# --- PATH SETUP ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# --- LOCAL IMPORTS ---
from src.analysis import processing as proc
from src.analysis import plotting as viz

# Global Storage for Final Comparison
ALL_RESULTS = {} 

# Common Snapshots
SNAPSHOTS_PRECISION = [10000, 100000, 1000000]
SNAPSHOTS_EXTENSION = [10000, 100000, 1000000, 10000000]

# --- PLOTTING STYLE ---
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

print(f"Project Root set to: {project_root}")
print("Modules loaded successfully.")

Project Root set to: /Users/ramsaydavis/dla
Modules loaded successfully.


Main Functions

In [18]:
def run_full_analysis(model_name, test_mode = False):
    """
    Runs the full computational pipeline for a specific model.
    Returns a dictionary containing all DataFrames and Grids.
    """
    limit = 10 if test_mode else None
    mode_str = 'TEST MODE (10 files)' if test_mode else 'FULL BATCH'
    print(f"nb_task: STARTING ANALYSIS FOR {model_name.upper()}...")
    
    # 1. Setup Paths
    # Assumes standard folder structure: results/analysis_clusters/{NAME}/...
    path_p = os.path.join(project_root, f'results/analysis_clusters/{model_name}/1M/*.npz')
    path_e = os.path.join(project_root, f'results/analysis_clusters/{model_name}/10M/*.npz')
    
    # 2. Run Metric Batches
    print(f"  > Running Precision Batch (N=1000)...")
    df_p = proc.batch_analysis(path_p, SNAPSHOTS_PRECISION, num_workers=2, limit=limit)
    
    print(f"  > Running Extension Batch (N=100)...")
    df_e = proc.batch_analysis(path_e, SNAPSHOTS_EXTENSION, num_workers=2, limit=limit)

    # 3. Generate Density Grid (N=10^6)
    print(f"  > Generating Density Grid...")
    grid, bins, max_r, rg = proc.generate_density_grid(
        file_pattern=path_p, 
        snapshot_N=1000000, 
        grid_size=1000,
        limit_files = limit
    )
    
    print(f"nb_task: COMPLETED {model_name}.")
    
    # Return a neat package
    return {
        'precision': df_p,
        'extension': df_e,
        'density_data': (grid, bins, max_r, rg)
    }

def display_model_results(model_name, results_dict):
    """
    Generates standard plots from the computed results.
    """
    data = results_dict
    
    # Unpack
    df_p = data['precision']
    df_e = data['extension']
    grid, bins, max_r, rg = data['density_data']
    
    # Plot 1: Anisotropy Evolution
    fig1, ax1 = plt.subplots(figsize=(8, 5))
    viz.plot_anisotropy_evolution(df_p, df_e, ax=ax1)
    ax1.set_title(f'{model_name}: Anisotropy Evolution')
    plt.show()

    # Plot 2: Density Map
    fig2 = viz.plot_density_map(
        grid, bins, 
        max_r_curve=max_r, 
        rg_curve=rg, 
        title=f"{model_name} Density (N=10^6)"
    )
    plt.show()
    
    # Print Stats
    final_score = df_e[df_e['snapshot_N'] == 10000000]['anisotropy_score'].mean()
    print(f"Final Anisotropy Score (N=10^7): {final_score:.4f}")

Analysis of Lattice DLA

In [19]:
ALL_RESULTS['Lattice'] = run_full_analysis('Lattice', test_mode=True)

nb_task: STARTING ANALYSIS FOR LATTICE...
  > Running Precision Batch (N=1000)...


TypeError: batch_analysis() got an unexpected keyword argument 'limit'

In [ ]:
display_model_results('Lattice', ALL_RESULTS['Lattice'])